In [1]:
#import pyspark
import findspark

In [2]:
findspark.find()

'D:\\Software\\spark-4.0.0-bin-hadoop3\\spark-4.0.0-bin-hadoop3'

In [3]:
#initiate spark
import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
c = pyspark.SparkConf().setAppName("test_app").setMaster("local")
sc = pyspark.SparkContext(conf = c)
spark = SparkSession(sc)

In [5]:
cust = spark.read.csv("D:\Data Engineering\Data Set\Custumer_Data.csv",inferSchema=None, header = True)

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\gunja\AppData\Local\Temp\ipykernel_12932\3933125708.py:1: SyntaxWarning: invalid escape sequence '\D'
  cust = spark.read.csv("D:\Data Engineering\Data Set\Custumer_Data.csv",inferSchema=None, header = True)


In [14]:
from pyspark.sql.functions import *

In [28]:
cust.show()

+---+--------+---+---------+----------+
| ID|    Name|Age|  Address|Wallet Bal|
+---+--------+---+---------+----------+
|  1|  Ramesh| 32|Ahmedabad|       200|
|  2|  Khilan| 23|    Delhi|      7500|
|  3| kaushik| 25|     Kota|     12000|
|  4|Chaitali| 22|   Mumbai|      6500|
|  5|  Hardik| 27|   Bhopal|       600|
|  6|   Komal| 26|       MP|       750|
|  7|   Muffy| 27|   Indore|       500|
+---+--------+---+---------+----------+



In [29]:
Orders = spark.read.csv("D:\Data Engineering\Data Set\Orders_Data.csv",inferSchema=None, header = True)

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\gunja\AppData\Local\Temp\ipykernel_12932\3300109379.py:1: SyntaxWarning: invalid escape sequence '\D'
  Orders = spark.read.csv("D:\Data Engineering\Data Set\Orders_Data.csv",inferSchema=None, header = True)


In [30]:
Orders.show()

+---+----------+---+------+
|OID|      Date| ID|Amount|
+---+----------+---+------+
|102|08-10-2009|  3|  3000|
|100|08-10-2009|  3|  1500|
|101|20-11-2009|  2|  1560|
|103|20-05-2008|  4|  2060|
+---+----------+---+------+



In [31]:
Orders.printSchema()

root
 |-- OID: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- ID: string (nullable = true)
 |-- Amount: string (nullable = true)



In [54]:
cust.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Wallet Bal: double (nullable = true)



In [55]:
cust = cust.withColumn("Wallet Bal", col("Wallet Bal").cast("double"))

In [56]:
cust.createOrReplaceGlobalTempView("cust_tb")


In [57]:
# spark.sql("Select * from cust_tb").show()
spark.sql("SELECT * FROM global_temp.cust_tb").show()


+---+--------+---+---------+----------+
| ID|    Name|Age|  Address|Wallet Bal|
+---+--------+---+---------+----------+
|  1|  Ramesh| 32|Ahmedabad|     200.0|
|  2|  Khilan| 23|    Delhi|    7500.0|
|  3| kaushik| 25|     Kota|   12000.0|
|  4|Chaitali| 22|   Mumbai|    6500.0|
|  5|  Hardik| 27|   Bhopal|     600.0|
|  6|   Komal| 26|       MP|     750.0|
|  7|   Muffy| 27|   Indore|     500.0|
+---+--------+---+---------+----------+



In [58]:
Orders.createOrReplaceGlobalTempView("Order_tb")

In [59]:
spark.sql("select a.Name, a.address , b.oid, b.Date from global_temp.cust_tb a right join global_temp.Order_tb b on a.id = b.id").show(5)

+--------+-------+---+----------+
|    Name|address|oid|      Date|
+--------+-------+---+----------+
| kaushik|   Kota|102|08-10-2009|
| kaushik|   Kota|100|08-10-2009|
|  Khilan|  Delhi|101|20-11-2009|
|Chaitali| Mumbai|103|20-05-2008|
+--------+-------+---+----------+



In [60]:
cust.join(Orders, cust['ID'] == Orders['ID']).show()

+---+--------+---+-------+----------+---+----------+---+------+
| ID|    Name|Age|Address|Wallet Bal|OID|      Date| ID|Amount|
+---+--------+---+-------+----------+---+----------+---+------+
|  2|  Khilan| 23|  Delhi|    7500.0|101|20-11-2009|  2|  1560|
|  3| kaushik| 25|   Kota|   12000.0|100|08-10-2009|  3|  1500|
|  3| kaushik| 25|   Kota|   12000.0|102|08-10-2009|  3|  3000|
|  4|Chaitali| 22| Mumbai|    6500.0|103|20-05-2008|  4|  2060|
+---+--------+---+-------+----------+---+----------+---+------+



In [61]:
cust.join(Orders, cust['ID'] == Orders['ID']).select("Name","Address","OID","Date").show()

+--------+-------+---+----------+
|    Name|Address|OID|      Date|
+--------+-------+---+----------+
|  Khilan|  Delhi|101|20-11-2009|
| kaushik|   Kota|100|08-10-2009|
| kaushik|   Kota|102|08-10-2009|
|Chaitali| Mumbai|103|20-05-2008|
+--------+-------+---+----------+



In [62]:
cust.join(Orders, cust['ID'] == Orders['ID'] , "Inner").select("Name","Address","OID","Date").show()

+--------+-------+---+----------+
|    Name|Address|OID|      Date|
+--------+-------+---+----------+
|  Khilan|  Delhi|101|20-11-2009|
| kaushik|   Kota|100|08-10-2009|
| kaushik|   Kota|102|08-10-2009|
|Chaitali| Mumbai|103|20-05-2008|
+--------+-------+---+----------+



In [63]:
#if having same col name in both table
cust.join(Orders, cust['ID'] == Orders['ID']).select(cust["Name"],cust["Address"],Orders["OID"],Orders["Date"]).show()

+--------+-------+---+----------+
|    Name|Address|OID|      Date|
+--------+-------+---+----------+
|  Khilan|  Delhi|101|20-11-2009|
| kaushik|   Kota|100|08-10-2009|
| kaushik|   Kota|102|08-10-2009|
|Chaitali| Mumbai|103|20-05-2008|
+--------+-------+---+----------+



In [64]:
cust.join(Orders, cust['ID'] == Orders['ID'], 'left').filter(col('Name') == 'kaushik').show()

+---+-------+---+-------+----------+---+----------+---+------+
| ID|   Name|Age|Address|Wallet Bal|OID|      Date| ID|Amount|
+---+-------+---+-------+----------+---+----------+---+------+
|  3|kaushik| 25|   Kota|   12000.0|100|08-10-2009|  3|  1500|
|  3|kaushik| 25|   Kota|   12000.0|102|08-10-2009|  3|  3000|
+---+-------+---+-------+----------+---+----------+---+------+



In [65]:
cust.join(Orders, cust['ID'] == Orders['ID'], 'left').orderBy(desc("Wallet Bal")).show()

+---+--------+---+---------+----------+----+----------+----+------+
| ID|    Name|Age|  Address|Wallet Bal| OID|      Date|  ID|Amount|
+---+--------+---+---------+----------+----+----------+----+------+
|  3| kaushik| 25|     Kota|   12000.0| 100|08-10-2009|   3|  1500|
|  3| kaushik| 25|     Kota|   12000.0| 102|08-10-2009|   3|  3000|
|  2|  Khilan| 23|    Delhi|    7500.0| 101|20-11-2009|   2|  1560|
|  4|Chaitali| 22|   Mumbai|    6500.0| 103|20-05-2008|   4|  2060|
|  6|   Komal| 26|       MP|     750.0|NULL|      NULL|NULL|  NULL|
|  5|  Hardik| 27|   Bhopal|     600.0|NULL|      NULL|NULL|  NULL|
|  7|   Muffy| 27|   Indore|     500.0|NULL|      NULL|NULL|  NULL|
|  1|  Ramesh| 32|Ahmedabad|     200.0|NULL|      NULL|NULL|  NULL|
+---+--------+---+---------+----------+----+----------+----+------+

